# RAG Pipeline — Data Visualization Course Assistant

**Domain:** Data Visualization course lecture notes (5 lectures: intro/goals, bar & column charts,
comparison-chart concepts, stacked & clustered charts, scatter/bubble/network charts).

**Track:** Core Track (text-only RAG).

This notebook builds and evaluates the retrieval-augmented generation pipeline end-to-end:
load → chunk → embed → store → retrieve → prompt → generate → evaluate → export.

Run cells top to bottom (`Kernel → Restart & Run All` should work cleanly).

In [ ]:
# Run once in your virtual environment (see Phase 0 of the project guide):
# pip install jupyter pandas numpy chromadb sentence-transformers pypdf ollama python-dotenv
#
# Also make sure Ollama is running locally and you've pulled a model, e.g.:
#   ollama pull llama3.2


## 2.0 Setup — imports & config

In [ ]:
import os
import re
import glob
import json
import uuid

import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
import ollama


In [ ]:
# --- Config (also written into backend/data/vector_store/config.json in the Export step,
# so the backend loads the SAME embedding model + settings used here) ---

DATA_DIR = "../data/raw_docs"                 # source documents
VECTOR_STORE_DIR = "./chroma_db"              # local dev store used while iterating in this notebook
EXPORT_VECTOR_STORE_DIR = "../backend/data/vector_store"  # final store the backend will load
COLLECTION_NAME = "lecture_notes"

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"     # small, fast, good enough for short lecture chunks
OLLAMA_MODEL = "llama3.2"                     # change to whatever model you've pulled locally

CHUNK_SIZE = 800     # characters
CHUNK_OVERLAP = 150  # characters
TOP_K = 4


## 2.1 Load & Inspect

**How many documents?** 5 source documents — one per lecture of the Data Visualization course.

**What formats?** All 5 originated as PowerPoint (`.pptx`) lecture decks and were extracted to
plain text (`.txt`), one file per lecture, with slide boundaries marked so the chunker can
respect natural section breaks.

**Which files failed to parse or need OCR?** None. All 5 decks were text-based (no scanned
images), so the text extracted cleanly with no OCR needed. Two lectures (3 and 5) contain a
number of image-only slides (chart screenshots) that carry no extractable text — those slides
contribute no content to the corpus, which is expected and fine for a text-only (Core Track)
pipeline.

In [ ]:
def load_documents(data_dir: str) -> list[dict]:
    docs = []
    for path in sorted(glob.glob(os.path.join(data_dir, "*.txt"))):
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        docs.append({"source": os.path.basename(path), "text": text})
    return docs


documents = load_documents(DATA_DIR)

for doc in documents:
    print(f"{doc['source']:35s} {len(doc['text']):6d} chars")

print(f"\nTotal documents: {len(documents)}")


## 2.2 Chunking Strategy

**Approach:** fixed-size chunking with overlap, but chunk boundaries are snapped to the
nearest slide separator (`---`) or whitespace rather than cutting mid-sentence, so a chunk
never splits a rule/definition in half.

**Why `CHUNK_SIZE = 800` characters (~150–180 words):** the lecture slides are short and
dense (one rule, one definition, or one table row per slide). 800 characters is roughly
2–3 slides' worth of content — enough to give the LLM a complete idea (e.g. a full "when to
use a bar chart" rule with its example questions) without pulling in unrelated slides from
later in the same lecture.

**Why `CHUNK_OVERLAP = 150` characters:** several slides build directly on the previous one
("Steps we must follow to achieve the goal" spans 3 consecutive slides in Lecture 1). A
~19% overlap keeps that continuity across chunk boundaries so a rule split across two slides
still appears whole in at least one chunk.

In [ ]:
def chunk_text(text: str, source: str, chunk_size: int, overlap: int) -> list[dict]:
    # Prefer splitting on the slide separator so we don't cut a slide in half.
    segments = [seg.strip() for seg in text.split("---") if seg.strip()]

    chunks = []
    buffer = ""
    for seg in segments:
        if len(buffer) + len(seg) + 1 <= chunk_size:
            buffer = f"{buffer}\n{seg}".strip()
        else:
            if buffer:
                chunks.append(buffer)
            # start new buffer, carrying over the tail of the previous buffer as overlap
            carry = buffer[-overlap:] if buffer else ""
            buffer = f"{carry}\n{seg}".strip()
            # if a single segment is itself bigger than chunk_size, hard-split it
            while len(buffer) > chunk_size:
                chunks.append(buffer[:chunk_size])
                buffer = buffer[chunk_size - overlap:]
    if buffer:
        chunks.append(buffer)

    return [
        {"chunk_id": f"{source}::{i}", "source": source, "text": c}
        for i, c in enumerate(chunks)
    ]


all_chunks = []
for doc in documents:
    doc_chunks = chunk_text(doc["text"], doc["source"], CHUNK_SIZE, CHUNK_OVERLAP)
    all_chunks.extend(doc_chunks)
    print(f"{doc['source']:35s} -> {len(doc_chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")


## 2.3 Embeddings & Vector Store

In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(texts, show_progress_bar=True).tolist()

print(f"Encoded {len(embeddings)} chunks into {len(embeddings[0])}-dim vectors")


In [ ]:
client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)

# Fresh collection each run of this notebook, so re-running doesn't duplicate chunks
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = client.create_collection(COLLECTION_NAME)

collection.add(
    ids=[c["chunk_id"] for c in all_chunks],
    documents=texts,
    embeddings=embeddings,
    metadatas=[{"source": c["source"]} for c in all_chunks],
)

print(f"Vector store now has {collection.count()} chunks persisted at '{VECTOR_STORE_DIR}'")


## 2.4 Retrieval & Prompting

In [ ]:
def retrieve(question: str, k: int = TOP_K) -> list[dict]:
    query_embedding = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=k)

    out = []
    for text, meta, dist in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        out.append({"text": text, "source": meta["source"], "distance": dist})
    return out


SYSTEM_PROMPT = (
    "You are a study assistant for a Data Visualization course. "
    "Answer the user's question using ONLY the context provided below. "
    "If the context does not contain the answer, say you don't know instead of guessing. "
    "Keep answers concise and cite which source(s) you used."
)


def build_prompt(question: str, chunks: list[dict]) -> str:
    context = "\n\n".join(
        f"[{i+1}] Source: {c['source']}\n{c['text']}" for i, c in enumerate(chunks)
    )
    return (
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        'Answer using only the context above. Reference the source number(s) you used, e.g. "(see [1])".'
    )


def ask_llm(question: str, chunks: list[dict], model: str = OLLAMA_MODEL) -> str:
    prompt = build_prompt(question, chunks)
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
    )
    return response["message"]["content"]


In [ ]:
# 10+ test questions covering all 5 lectures
TEST_QUESTIONS = [
    "What is the goal of this data visualization course, in terms of datasets, questions, and answers?",
    "What are the 6 steps we should follow to go from a dataset to an answer?",
    "What is the difference between a drag/drop-driven approach and a code-driven approach?",
    "When should I use a bar chart instead of a column chart?",
    "What are the guidelines for drawing a good bar chart?",
    "What is 'chart junk' and why does it matter?",
    "What is the difference between magnitude and cardinality?",
    "When should I use a stacked chart versus a clustered chart?",
    "What color strategy should I use for a stacked chart?",
    "When is a scatter plot the right choice for a dataset?",
    "What is the difference between correlation and cardinality?",
    "What is overplotting, and how can jittering or binning help with it?",
]

print(f"{len(TEST_QUESTIONS)} test questions defined")


In [ ]:
# Quick retrieval sanity check for all questions BEFORE spending time on LLM calls
for q in TEST_QUESTIONS:
    top = retrieve(q, k=1)[0]
    print(f"Q: {q}\n  -> top match: {top['source']} (distance={top['distance']:.3f})\n")


## 2.5 Vision Component

**Not applicable.** This project follows the **Core Track** (text-only RAG). No
Computer Vision / YOLO component is included in this pipeline.

## 2.6 Evaluation

Running each test question through the full pipeline (retrieve → prompt → local LLM) and
recording the retrieved source, the generated answer, and a manual correctness judgement.

**Read this before running:** the `correct` column and the "failure cases" write-up below are
placeholders. You need to run this cell yourself with Ollama running locally, actually read
each generated answer against the lecture content, and fill in `True`/`False` honestly — this
is the part instructors will probe in the live demo, so the results need to be real, not
guessed.

In [ ]:
eval_rows = []
for q in TEST_QUESTIONS:
    chunks = retrieve(q, k=TOP_K)
    answer = ask_llm(q, chunks)
    eval_rows.append({
        "question": q,
        "retrieved_sources": ", ".join(sorted({c["source"] for c in chunks})),
        "answer": answer,
        "correct": None,  # <-- fill in True/False after reading the answer yourself
    })

eval_df = pd.DataFrame(eval_rows)
eval_df


In [ ]:
# After you've manually filled in the "correct" column above, re-run this cell to save it.
eval_df.to_csv("evaluation_results.csv", index=False)
print("Saved evaluation_results.csv")
print(f"Accuracy so far: {eval_df['correct'].mean() if eval_df['correct'].notna().any() else 'not yet scored'}")


**Failure cases (fill in after running):** Note the specific questions where the answer
was wrong or hallucinated, whether the *retrieval* step pulled the wrong chunk (bad recall)
or the *generation* step ignored good context, and what you changed as a result — e.g. a
larger `TOP_K`, a stricter system prompt, or a smaller chunk size. Write this from your own
observed output, not from what you expect to happen.

## 2.7 Export — persist the store for the backend

In [ ]:
import shutil

# Copy the notebook's vector store into the location the FastAPI backend loads from
if os.path.exists(EXPORT_VECTOR_STORE_DIR):
    shutil.rmtree(EXPORT_VECTOR_STORE_DIR)
shutil.copytree(VECTOR_STORE_DIR, EXPORT_VECTOR_STORE_DIR)

# Save the config the backend needs to reproduce the exact same embedding space
config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "collection_name": COLLECTION_NAME,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
}
with open(os.path.join(EXPORT_VECTOR_STORE_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print(f"Exported vector store + config to {EXPORT_VECTOR_STORE_DIR}")


### Done

Next: start the backend (`uvicorn app.main:app --reload` from `backend/`) and the frontend
(`streamlit run app.py` from `frontend/`) — see the root README for full setup steps.